## Consistency Check

**Information Need:** Understand whether a selected attribute has a value that complies to a specified rule across all events in the same case.

**Motivation:** During initial analysis, analysts may formulate expectations about how recorded attribute values should behave within a case. These expectations can be expressed as domain-specific rules concerning individual attributes or relationships between multiple attributes. For example, an attribute may be expected to satisfy a value constraint, remain constant across events, or depend on the values of other attributes. Checking compliance with such rules allows analysts to validate their expectations and identify inconsistencies in the recorded information.

**Preconditions:** The rule to be checked and the attributes to which it applies are specified.

**Approach:** For each case, evaluate the specified rule over the relevant attribute values and events, identifying the cases for which the rule does not hold.

**Output:** An assessment of the compliance of the domain-specific rule within each case, together with the identifiers of the cases that violate it.

*Note: Each check requires custom code capturing the specific rule.*

In [ ]:
from json.decoder import NaN
import pm4py

# --- Configuration -----------------------------------------------------------

LOG_PATH = "../../data/SepsisCases2020EventLog.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)
  
display(event_log.head())

### Pattern execution

In [ ]:
# Code for checking the rule.
# Each check requires custom code capturing the specific relationship expected among the selected attributes.
# Redefine this function for the rule and attributes relevant to the loaded event log.
# It shall return only the rows for which the rule is violated.

# Relevant for the Sepsis event log: attribute SIRSCriteria2OrMore is expected to be True if at least two of
# the attributes SIRSCritHeartRate, SIRSCritLeucos, SIRSCritTachypnea, and SIRSCritTemperature are True.
def check_rule(df):
    expected = (df['SIRSCritHeartRate'] + df['SIRSCritLeucos']
        + df['SIRSCritTachypnea'] + df['SIRSCritTemperature']) >= 2
    return df[expected != df['SIRSCriteria2OrMore']]

In [ ]:
violations = check_rule(event_log)

# Relevant for the Sepsis event log: display only the columns relevant to the rule check.
selected_columns = [col for col in event_log.columns if col.startswith('SIRS')]
display(violations[violations[ACTIVITY] == 'ER Registration'][selected_columns])

### Another example

In [ ]:
# --- Configuration -----------------------------------------------------------

LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

display(event_log.head())

In [ ]:
# Code for checking the rule.
# Each check requires custom code capturing the specific relationship expected among the selected attributes.
# Redefine this function for the rule and attributes relevant to the loaded event log.
# It shall return only the cases for which the rule is violated.

# Relevant for the Road Traffic Fine Management event log: within each case, the total amount paid
# (last value of totalPaymentAmount) is expected to not exceed the fine that is due, i.e. the last
# value of amount (the base fine, possibly increased by a penalty) plus the mailing expense.
def check_rule(df):
    def last_value(series):
        series = series.dropna()
        return series.iloc[-1] if len(series) else float("nan")

    per_case = df.sort_values(TIMESTAMP).groupby(CASE_ID).agg(
        total_paid=("totalPaymentAmount", last_value),
        amount_due=("amount", last_value),
        expense=("expense", last_value),
    )

    has_payment = per_case["total_paid"].notna()
    within_limit = per_case["total_paid"] <= per_case["amount_due"] + per_case["expense"].fillna(0)
    return per_case[has_payment & ~within_limit]

In [ ]:
violations = check_rule(event_log)

display(violations)